# Build a Scalable Multi-Agent AI System with Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_planner_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Welcome to this hands-on tutorial on building intelligent, scalable AI agents!

## What We'll Cover

1. **Agent Design Patterns** - How to build modular, specialized AI agents
2. **Dynamic Task Planning** - Using LLMs to create execution plans with dependencies
3. **Parallel Execution** - The "fanout" pattern for concurrent task execution
4. **Result Propagation** - Passing outputs from one agent to dependent agents
5. **Production Deployment** - Scaling agents with Flyte's distributed architecture

---

## 🏗️ Planner AI Agent System

```
User Request
    ↓
┌─────────────────┐
│ Planner Agent   │ ← Analyzes request, creates execution plan
└────────┬────────┘
         │
         ├─→ Step 0: math (deps: [])     ┐
         ├─→ Step 1: string (deps: [])   ├─ Wave 1: Parallel execution
         ├─→ Step 2: search (deps: [])   ┘
         │
         └─→ Step 3: code (deps: [0,1,2]) ← Wave 2: Waits for 0,1,2
```

**Key Components:**
- 🧠 **Planner Agent**: Routes tasks & identifies dependencies
- 🔧 **Specialist Agents**: Math, String, Web Search, Code
- **Tools** 
- 🎯 **Orchestrator**: Executes plan with parallel fanout
- 🔗 **Dependency Engine**: Passes results between agents

---

## ⚙️ Setup and Configuration
- Scales horizontally using Flyte's distributed execution

Think of it as a **smart task coordinator** that knows when to fan out work and when to wait for dependencies.

This notebook will guide you through the process of building and deploying a multi-agent system using Flyte.
In this case the project is structured as a Python package, you can find the code with subfolders in the `tutorials/multi-agent-workflows/` directory.
This notebook will read in the most relevant code files and display them for your reference. but to make changes use the code editor in your IDE.

**What's happening here:**

This config file sets up:
- 📦 **Base Environment** (`base_env`): Shared Docker image + secrets for all agents
- 🔑 **API Keys**: OpenAI credentials loaded from environment
- 🐳 **Container Image**: Debian base with Python dependencies

**Why this matters:** Each agent runs in its own Flyte task, but they share this base configuration. This means you can easily scale specific agents independently while maintaining consistent dependencies.

In [19]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [20]:
view_file("requirements.txt")

In [24]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

- If you don't have a Flyte cluster you can request access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting the endpoint


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [17]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


## Run the Agent

At this point you should be setup to run the planner agent. I suggest giving it a try!

We'll walk through the code in the next section

**Run locally:**

**Run on the remote Flyte cluster:**

# Code walk through

## 🛠️ Utility Functions

Before we dive into agents and tools, let's look at the infrastructure that makes our system work:

### **Decorators** - Making Agent/Tool Creation Easy

Our decorator system provides framework-like capabilities:
- `@agent("name")` - Registers agents in a global registry
- `@tool(agent="name")` - Associates tools with specific agents
- Automatic discovery and routing

**Think of it like Flask routes or FastAPI endpoints** - decorators make complex registration simple.

In [10]:
view_file("utils/decorators.py")


### **Plan Executor**

This is where the magic happens:
- Takes a user task (e.g., "Calculate 5 + 3")
- Asks an LLM to create a JSON plan with tool calls
- Executes each tool step-by-step
- Handles the "previous" result pattern for chaining

**Key Innovation:** Few-shot prompting ensures the LLM returns valid JSON tool plans.


In [11]:
view_file("utils/plan_executor.py")

Planner Agent

In [22]:
view_file("agents/planner_agent.py")

---

## 🔧 Tools: Giving Agents Capabilities

Tools are **the actions agents can take**. Think of them as the agent's hands - without tools, an agent can only think and talk. With tools, it can actually DO things.

### What's a Tool?

A tool is a simple async function that does one thing well:

```python
from utils.decorators import tool
import flyte

@tool(agent="math")
@flyte.trace
async def add(a: float, b: float) -> float:
    """Add two numbers together"""
    return a + b
```

**Three things happen here:**

1. `@tool(agent="math")` - Registers this tool with the math agent
2. `@flyte.trace` - Makes execution visible in Flyte UI
3. `async def` - Enables parallel execution

### How Tools Get Called

When an agent needs to solve a task, the LLM generates a plan like:
```json
[
  {"tool": "add", "args": [2, 3], "reasoning": "Adding 2 and 3"}
]
```

The plan executor:
1. Finds the `add` tool in the registry
2. Calls it with arguments: `add(2, 3)`
3. Returns the result: `5`

**Why async?** Multiple tools can run in parallel! If an agent needs to add AND multiply, both operations can happen simultaneously.

### Our Toolkit

We have four specialized toolkits, each designed for a specific agent:

- 📐 **Math Tools** (`tools/math_tools.py`) - add, subtract, multiply, divide, power, factorial
- ✂️ **String Tools** (`tools/string_tools.py`) - word_count, letter_count
- 🔍 **Web Search Tools** (`tools/web_search_tools.py`) - duck_duck_go, fetch_webpage
- 💻 **Code Tools** (`tools/code_tools.py`) - execute_python

**Why separate files?** Each agent only gets the tools it needs. The math agent doesn't need web search tools cluttering its context.

Let's look at each toolkit:

### Math Tools

**Capabilities:** Basic arithmetic, exponents, factorials

These are the building blocks for mathematical reasoning. Simple, focused, composable.

In [6]:
print_code_file("tools/math_tools.py")


### Web Search Tools

**Capabilities:** DuckDuckGo search, webpage content fetching

Gives agents access to real-time information from the web. Notice the adjustable parameters (region, time filter, etc.) - this lets the LLM customize searches.

In [25]:
view_file("tools/web_search_tools.py")


---

## 🤖 Agents: Specialists That Use Tools to Solve Problems

Each agent follows the same pattern:

### Agent Anatomy:
```python
@dataclass
class MathAgentResult:
    """Structured output - type-safe and serializable"""
    final_result: str
    steps: str
    error: str = ""

env = base_env  # Shared Flyte environment

@env.task              # Makes this a Flyte task (containerized, scalable)
@agent("math")         # Registers in agent registry
async def math_agent(task: str) -> MathAgentResult:
    """
    1. Receives a task (e.g., "Calculate 5 factorial")
    2. Asks LLM to create a tool execution plan
    3. Executes the plan using registered math tools
    4. Returns structured result
    """
    result = await execute_plan(task, agent="math", system_msg=...)
    return MathAgentResult(final_result=result["final_result"], ...)
```

### Key Design Decisions:

**Why Flyte tasks?**
- Each agent runs in its own container
- Independent scaling (e.g., 10 math agents, 2 web search agents)
- Resource isolation and monitoring

**Why dataclasses?**
- Type-safe outputs (no dict key errors)
- Flyte-native serialization (no pickle issues)
- Clear contracts between agents

**Why async?**
- Parallel tool execution
- Non-blocking I/O for web searches
- Better resource utilization

Let's examine each specialist agent:

### Math Agent

**Specialty:** Arithmetic operations, exponents, factorials

**How it works:**
1. User asks: "Calculate 5 factorial"
2. LLM generates plan: `[{"tool": "factorial", "args": [5], "reasoning": "Calculating 5!"}]`
3. Plan executor finds and calls `factorial(5)` 
4. Tool executes: `5! = 120`
5. Returns: `MathAgentResult(final_result="120", steps="[...]")`

**Example Flow:**
```
Input: "Calculate 2 to the power of 8"
  ↓
LLM Plan: [{"tool": "power", "args": [2, 8]}]
  ↓
Tool Call: power(2, 8)
  ↓
Result: "256"
```

Simple, focused, reliable. The agent knows nothing about web search or string manipulation - it's a specialist!

In [26]:
view_file("agents/math_agent.py")

### Web Search Agent

**Specialty:** Real-time information retrieval from the web

**Simple search example:**
```
Input: "Search for Python async tutorials"
  ↓
LLM Plan: [{"tool": "duck_duck_go", "args": ["Python async tutorial", 5, "us-en", "moderate", null]}]
  ↓
Tool Call: duck_duck_go("Python async tutorial", max_results=5, ...)
  ↓
Result: List of 5 search results with titles, URLs, and snippets
```

**Advanced multi-step example:**
```
Input: "Search for Flyte documentation and fetch the homepage content"
  ↓
LLM Plan: [
  {"tool": "duck_duck_go", "args": ["Flyte documentation", 3, ...]},
  {"tool": "fetch_webpage", "args": ["https://docs.flyte.org", 3000]}
]
  ↓
Step 1: Get search results
Step 2: Fetch actual webpage content
  ↓
Result: Full webpage text from Flyte docs
```

**Power of tool chaining:** The agent can search first, then fetch details from specific URLs in the results. The LLM decides the strategy!

In [27]:
view_file("agents/web_search_agent.py")


### Planner Agent - The Brain of the System 🧠

**This is where the magic happens!**

The Planner Agent is unique - instead of using tools, it **coordinates other agents**:

#### What It Does:
1. **Analyzes** the user's request
2. **Identifies** which agents are needed
3. **Determines dependencies** between steps
4. **Creates an execution plan** with parallel opportunities

#### Example:

**User Request:**
```
"Calculate 2+3 and 5+6, then add those results together"
```

**Planner's Output:**
```python
PlannerDecision(steps=[
    AgentStep(agent="math", task="Calculate 2+3", dependencies=[]),      # Step 0
    AgentStep(agent="math", task="Calculate 5+6", dependencies=[]),      # Step 1
    AgentStep(agent="math", task="Add results", dependencies=[0, 1])     # Step 2
])
```

**Key Insight:** Steps 0 and 1 have `dependencies=[]`, so they can run **in parallel**. Step 2 depends on both, so it waits and receives their results.

#### The Prompt Engineering Magic:

The planner uses **few-shot learning** to teach the LLM about dependencies:

```python
system_msg = """You are a task planner. Analyze requests and create execution plans.

Available agents:
- math: arithmetic, factorials, exponents
- string: word count, letter count
- web_search: search web, fetch pages
- code: execute Python code

CRITICAL: Identify dependencies!
- If step 2 needs results from step 0: dependencies=[0]
- If step 3 needs steps 0 and 1: dependencies=[0,1]
- Independent steps: dependencies=[]

Example:
Request: "Calculate 2+3 and 5+6, then multiply results"
Response:
{
  "steps": [
    {"agent": "math", "task": "2+3", "dependencies": []},
    {"agent": "math", "task": "5+6", "dependencies": []},
    {"agent": "math", "task": "multiply results", "dependencies": [0,1]}
  ]
}
"""
```

The LLM learns from examples to create dependency-aware plans! This enables:
- ✅ **Automatic parallelization** of independent tasks
- ✅ **Smart dependency chains** for sequential work
- ✅ **Dynamic DAG generation** - the execution graph is created at runtime from natural language

This is **intelligent task routing** - no hardcoded workflows needed!

---

## 🎯 The Orchestrator: Bringing It All Together

This is the **execution engine** that makes parallel, dependency-aware execution possible.

### How It Works:

#### 1. **Call the Planner**
```python
planner_decision = await planner_agent(user_request)
# Returns: PlannerDecision with steps and dependencies
```

#### 2. **Build Execution Waves**
```python
while pending_steps:
    # Find steps with satisfied dependencies
    ready_steps = [step for step in pending if all deps completed]
    
    # Execute them ALL IN PARALLEL
    results = await asyncio.gather(*[execute(step) for step in ready_steps])
```

#### 3. **Pass Results to Dependent Steps**

When a step has dependencies, we augment its task with context from previous results:
```python
if step.dependencies:
    context = "Results from previous steps:\n"
    for dep_id in step.dependencies:
        context += f"Step {dep_id}: {completed_results[dep_id]}\n"
    
    task = context + f"\nYour task: {step.task}"
    
    # Now the agent knows what to work with!
    result = await get_agent(step.agent)(task)
```

**This is automatic result passing** - no manual wiring needed! The orchestrator injects previous results into the agent's prompt, so it has the context it needs to complete its task.

### Visual Example:

```
User: "Calculate 2+3 and 5+6, then add results"

Planner → [Step 0: 2+3 (deps:[]), Step 1: 5+6 (deps:[]), Step 2: add (deps:[0,1])]

╔═══════════════════════════════════════╗
║           WAVE 1 (Parallel)           ║
╠═══════════════════════════════════════╣
║  Step 0: math_agent("Calculate 2+3")  ║ → Result: "5"
║  Step 1: math_agent("Calculate 5+6")  ║ → Result: "11"
╚═══════════════════════════════════════╝
              ↓ (both complete)
╔═══════════════════════════════════════╗
║            WAVE 2 (Sequential)        ║
╠═══════════════════════════════════════╣
║  Step 2: math_agent(                  ║
║    "Results from previous steps:      ║
║     Step 0: 5                         ║
║     Step 1: 11                        ║
║                                       ║
║     Your task: Add results"           ║
║  )                                    ║ → Result: "16"
╚═══════════════════════════════════════╝
```

Notice how Step 2 receives the results from Steps 0 and 1 automatically! The orchestrator builds the context string and the math agent sees both results in its prompt.

### Key Features:

✅ **Automatic Parallelization** - Uses `asyncio.gather()` for fanout  
✅ **Dependency Resolution** - Tracks which steps are ready  
✅ **Result Propagation** - Injects previous results into dependent tasks  
✅ **Error Handling** - Gracefully handles circular dependencies  
✅ **Observable** - Full logging of each wave and step  

This is essentially a **mini workflow engine** built on Flyte's distributed execution primitives!

Let's see the code:

In [28]:

view_file("workflows/planner.py")

---

## 🚀 Running the Workflow

The workflow supports two execution modes:

### 🌐 Remote Execution (Production Mode)
```bash
python -m workflows.planner
```
**What happens:**
- Uses `flyte.init_from_config()` to connect to your Flyte cluster
- Each agent runs in its own containerized task
- Parallel execution happens across multiple workers
- Full observability in Flyte UI with execution graphs
- Scalable and production-ready

**When to use:** Production deployments, large workloads, distributed execution

---

### 💻 Local Execution (Development Mode)
```bash
python -m workflows.planner --local
```
**What happens:**
- Uses `flyte.init()` for in-process execution
- All agents run locally (no cluster needed)
- Still uses async/parallelization via asyncio
- Faster iteration for development and testing
- Same code paths, different execution backend

**When to use:** Local development, quick testing, debugging

---

### 📝 Example Test Prompts

Try these in the notebook or by editing `workflows/planner.py`:

#### Simple:
```python
"Calculate 5 factorial"
```

#### Parallel Execution:
```python
"Calculate 2 plus 3 and 5 plus 6, then add those results together"
# → Wave 1: [2+3, 5+6] in parallel
# → Wave 2: [add 5 + 11]
```

#### Mixed Agents:
```python
"Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result"
# → Wave 1: [math: 10*5, string: count] in parallel
# → Wave 2: [math: 50 * 2 = 100]
```

#### Complex Multi-Agent:
```python
"Calculate 5 factorial, count letters in 'hello', and search for 'Flyte', then write Python code to combine them"
# → Wave 1: [math, string, web_search] all in parallel
# → Wave 2: [code agent receives all 3 results]
```

Let's run it!

In [ ]:
!python -m workflows.planner

---

## 🎓 Key Takeaways

Congratulations! You've just explored a production-ready multi-agent system. Here's what makes it special:

### Architecture Highlights:

1. **🧩 Modular Design**
   - Each agent is self-contained with its own tools
   - Easy to add new agents (just create file + import in planner)
   - Clear separation of concerns

2. **⚡ Parallel Execution**
   - Automatic fanout for independent tasks
   - Uses `asyncio.gather()` for concurrency
   - Leverages Flyte's distributed execution

3. **🔗 Dependency Management**
   - LLM determines dependencies at runtime
   - Results automatically passed to dependent steps
   - Circular dependency detection

4. **🎯 Smart Routing**
   - Planner analyzes requests in natural language
   - Creates optimal execution plans
   - No hardcoded workflows!

5. **📊 Production Ready**
   - Type-safe with dataclasses
   - Observable with Flyte UI
   - Scalable across multiple workers
   - Error handling and logging

### What You Can Build:

- **Data pipelines** with conditional branching
- **Research assistants** that search + analyze + summarize in parallel
- **Code generation systems** that plan → implement → test concurrently
- **Multi-modal applications** combining vision, text, and code agents

### Next Steps:

1. **Add your own agents** - Create specialized agents for your domain
2. **Extend tools** - Add more capabilities (database access, API calls, etc.)
3. **Improve prompting** - Fine-tune the planner for better dependency detection
4. **Add memory** - Implement cross-execution context
5. **Deploy** - Scale to production with Flyte's cluster management

---

## 🙏 Thank You!

Questions? Try experimenting with different prompts and watch how the system adapts!

**Remember:** The power is in the **dynamic DAG generation**. The LLM creates the execution plan, and Flyte executes it efficiently. That's the magic! ✨